In [6]:
from google.colab import userdata
API_KEY = userdata.get('GNANIAPIKEY')

In [17]:
import requests
import json
from google.colab import files, userdata
from IPython.display import Audio, display

# 🔑 Load your API key securely
API_KEY = userdata.get('GNANIAPIKEY')
BASE_URL = "https://api.vachana.ai"

HEADERS_KEY = {
    "X-API-Key-ID": API_KEY
}

# ------------------------------------------------------------------
# PART 1: SPEECH-TO-TEXT (Prisma v2.5)
# ------------------------------------------------------------------
def transcribe_audio_file(file_path, language_code="hi-IN"):
    url = f"{BASE_URL}/stt/v3"

    with open(file_path, "rb") as audio_file:
        files_payload = {"audio_file": (file_path, audio_file)}
        data_payload = {
            "language_code": language_code,
            "preferred_language": language_code,
            "format": "transcribe",
            "itn_native_numerals": "true"
        }

        print("⏳ Transcribing patient audio with Prisma v2.5...")
        response = requests.post(url, headers=HEADERS_KEY, data=data_payload, files=files_payload)

    if response.status_code == 200:
        result = response.json()
        transcript = result.get("transcript") or result.get("text") or str(result)
        return transcript
    else:
        print(f"❌ STT Error {response.status_code}: {response.text}")
        return None

# ------------------------------------------------------------------
# PART 2: ADVANCED CLINICAL TRIAGE & SOLUTIONS ENGINE
# ------------------------------------------------------------------
def evaluate_symptoms_and_solutions(transcript):
    if not transcript:
        return "UNKNOWN", "विवरण नहीं मिला।", []

    t_lower = transcript.lower()

    # 1. Emergency Check (Red Flags - English & Hindi)
    emergency_keywords = ["chest pain", "breathlessness", "unconscious", "blue lips", "heavy bleeding", "saans", "सांस", "छाती में दर्द"]
    if any(w in t_lower for w in emergency_keywords):
        triage_level = "RED (Emergency - Immediate Hospitalization)"
        advice = "चेतावनी: गंभीर लक्षण पाए गए हैं। तुरंत एम्बुलेंस को कॉल करें या नजदीकी अस्पताल के आपातकालीन कक्ष में जाएं।"
        solutions = [
            "Call local emergency services or 108 immediately.",
            "Do not leave the patient unattended.",
            "Keep the patient calm and seated in a comfortable posture."
        ]

    # 2. Fever & Body Pain (Yellow / Routine - English & Hindi)
    elif any(w in t_lower for w in ["bukhar", "fever", "bukhar hai", "dard", "body pain", "बुखार", "दर्द", "बदन दर्द"]):
        triage_level = "YELLOW (Moderate - Outpatient Consultation)"
        advice = "आपके लक्षणों के अनुसार आपको बुखार है। कृपया पर्याप्त मात्रा में पानी पिएं, आराम करें और डॉक्टर से सलाह लें।"
        solutions = [
            "Monitor body temperature every 4-6 hours.",
            "Stay hydrated with oral rehydration salts (ORS) or water.",
            "Consult a general physician within 24 hours for proper medication."
        ]

    # 3. Stomach / Digestive Issues (Yellow / Routine - English & Hindi)
    elif any(w in t_lower for w in ["pet", "stomach", "vomit", "ulti", "loose motion", "पेट", "उल्टी", "दस्त", "पेट दर्द"]):
        triage_level = "YELLOW (Moderate - Digestive Care)"
        advice = "पेट की समस्या के लिए हल्का और ताजा भोजन लें। ओआरएस का घोल लें और डॉक्टर से मिलें।"
        solutions = [
            "Consume light, easily digestible foods like khichdi or boiled rice.",
            "Avoid spicy, oily, or outside food.",
            "Drink plenty of boiled and filtered water or ORS solution."
        ]

    # 4. General / Default Routine Care
    else:
        triage_level = "GREEN (Mild - Self-Care & Observation)"
        advice = "नमस्ते। अपने स्वास्थ्य पर नजर रखें, पर्याप्त विश्राम करें और यदि लक्षण बने रहें तो स्वास्थ्य केंद्र जाएं।"
        solutions = [
            "Get adequate rest for 1-2 days.",
            "Maintain proper hydration and eat nutritious meals.",
            "Visit a primary health center if symptoms worsen."
        ]

    return triage_level, advice, solutions

# ------------------------------------------------------------------
# PART 3: TEXT-TO-SPEECH (Timbre v2.5)
# ------------------------------------------------------------------
def synthesize_voice(text, language_code="hi-IN", output_filename="response.wav"):
    url = f"{BASE_URL}/api/v1/tts/inference"
    headers_json = {**HEADERS_KEY, "Content-Type": "application/json"}

    payload = {
        "text": text,
        "voice": "Nalini",
        "model": "timbre-v2.5",
        "language": language_code,
        "speed": 1.0,
        "audio_config": {
            "sample_rate": 24000,
            "container": "wav"
        }
    }

    response = requests.post(url, headers=headers_json, json=payload)

    if response.status_code == 200:
        with open(output_filename, "wb") as f:
            f.write(response.content)
        return output_filename
    else:
        print(f"❌ TTS Error: {response.status_code} - {response.text}")
        return None

# ------------------------------------------------------------------
# RUNNABLE INTERACTIVE FLOW
# ------------------------------------------------------------------
def run_live_health_bot():
    print("📂 Please upload your patient audio file (.wav or .mp3):")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded.")
        return

    audio_filename = list(uploaded.keys())[0]
    print(f"\n✅ File uploaded: {audio_filename}\n")

    # Transcribe speech
    patient_text = transcribe_audio_file(audio_filename, language_code="hi-IN")
    print(f"🗣️ Transcribed Patient Speech: {patient_text}\n")

    if not patient_text:
        return

    # Evaluate triage & get tailored solutions
    triage_level, response_text, actionable_solutions = evaluate_symptoms_and_solutions(patient_text)

    print(f"🏥 Assessed Triage Level: {triage_level}")
    print(f"💬 Spoken Advice to Patient: {response_text}\n")

    print("📋 Actionable Medical Solutions / Next Steps for Health Worker:")
    for i, sol in enumerate(actionable_solutions, 1):
        print(f"   {i}. {sol}")
    print("\n")

    # Generate speech for the advice
    output_audio = synthesize_voice(response_text, language_code="hi-IN")

    if output_audio:
        print("🔊 Playing Generated Voice Response:")
        display(Audio(output_audio, autoplay=True))

run_live_health_bot()

📂 Please upload your patient audio file (.wav or .mp3):


Saving voicetest.wav to voicetest (3).wav

✅ File uploaded: voicetest (3).wav

⏳ Transcribing patient audio with Prisma v2.5...
🗣️ Transcribed Patient Speech: हेलो मेरा नाम मोहम्मद गुलाम मुर्शिद है मेरा मुझे २ दिन से बुखार है और मेरा पेट में भी थोड़ा थोड़ा प्रॉब्लम हो रहा है थोड़ा मुझे क्या करना चाहिए कौन सा मेडिसिन लेना चाहिए थोड़ा बोल दीजिए

🏥 Assessed Triage Level: YELLOW (Moderate - Outpatient Consultation)
💬 Spoken Advice to Patient: आपके लक्षणों के अनुसार आपको बुखार है। कृपया पर्याप्त मात्रा में पानी पिएं, आराम करें और डॉक्टर से सलाह लें।

📋 Actionable Medical Solutions / Next Steps for Health Worker:
   1. Monitor body temperature every 4-6 hours.
   2. Stay hydrated with oral rehydration salts (ORS) or water.
   3. Consult a general physician within 24 hours for proper medication.


🔊 Playing Generated Voice Response:
